In [ ]:
# [로컬 mobius4용 사본] 원본(세종대 원격 Mobius)에서 바꾼 곳은 셋뿐이다:
#   ① 접속 주소 → 로컬 mobius4  ② AE 이름 → TS-train / TS-test
#   ③ 헤더: Origin을 각자의 관리자 신원(ADMIN_ORIGIN 변수)으로, 세종대 전용 확장 헤더는 제거
import csv
import requests
import os
import numpy as np
import torch
import torch.nn as nn
import json
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset
from urllib.parse import urlparse

# --------------------------------------------------------------------------------
# (1) 시드 고정 (재현성)
# --------------------------------------------------------------------------------
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

# --------------------------------------------------------------------------------
# (2) 모델 파라미터 
# --------------------------------------------------------------------------------
hidden_size   = 5
window_size   = 3
batch_size    = 4
num_epochs    = 100
learning_rate = 0.001

# --------------------------------------------------------------------------------
# (3) Mobius 플랫폼 설정
# --------------------------------------------------------------------------------
MOBIUS_BASE_URL    = "http://localhost:7599/Mobius/"
MOBIUS_TRAIN_NAME     = "TS-train"
MOBIUS_TEST_NAME   = "TS-test"

# ★ 각자 수정: 자기 mobius4의 관리자 신원(config/local.json의 cse.admin 값).
#   틀리면 오류 없이 빈 목록을 받아 학습 데이터가 0건이 된다.
ADMIN_ORIGIN       = "SM2"

MOBIUS_TRAIN_AE_URL  = os.path.join(MOBIUS_BASE_URL, MOBIUS_TRAIN_NAME)
MOBIUS_TEST_AE_URL  = os.path.join(MOBIUS_BASE_URL, MOBIUS_TEST_NAME)
HEADERS_GET = {
    'Accept':       'application/json',
    'X-M2M-RI':     '12345',
    'X-M2M-Origin': ADMIN_ORIGIN
}

# --------------------------------------------------------------------------------
# (4) 센서 컨테이너 이름
# --------------------------------------------------------------------------------
CONTAINERS = {
    "precipitation": "precipitation_sensor",
    "temperature":   "temperature_sensor",
    "wind":          "wind_sensor",
    "ground_truth":  "ground_truth"
}

# --------------------------------------------------------------------------------
# (5) HTTP 요청 및 데이터 가져오기 
# --------------------------------------------------------------------------------
def http_get(url, params=None, headers=None, iotPlatform=None):
    """
    주어진 URL에 GET 요청을 보내고, 결과를 JSON(dict)로 반환한다.
    iotPlatform=True이면 OneM2M용 기본 헤더가 적용된다.
    """
    if iotPlatform:
        headers = {
            'Accept':       'application/json',
            'X-M2M-RI':     '12345',
            'X-M2M-Origin': ADMIN_ORIGIN
        }

    try:
        response = requests.get(url, params=params, headers=headers, timeout=10)
        response.raise_for_status()
        return json.loads(response.text)

    except requests.ConnectTimeout:
        print(f"Connection timed out for URL: {url}")
        return None

    except requests.HTTPError as http_err:
        print(f"HTTP error occurred for URL {url}: {http_err}")
        return None

    except Exception as err:
        print(f"An error occurred for URL {url}: {err}")
        return None

def all_cin_get_uri(path, max_retries=10):
    """
    주어진 컨테이너 경로에서 모든 콘텐츠 인스턴스(CIN) URI를 얻어
    각각을 GET하여 'con' 필드를 모아 리스트로 반환한다.
    """
    path = path + '?fu=1&ty=4'  # 모든 CIN(ResourceType=4) 조회 쿼리
    parsed_path = urlparse(path)
    base_path   = f"{parsed_path.scheme}://{parsed_path.netloc}/"
    resource_path = path.split('?')[0]

    con_list = []
    print(f"  [DISCOVERY] 대상 경로: {resource_path}")
    all_uri  = http_get(path, iotPlatform=True)
    if not all_uri:
        print(f"  [DISCOVERY] URI 목록을 가져오지 못했습니다: {resource_path}")
        return con_list  # 에러 시 빈 리스트 반환

    uri_list = all_uri.get("m2m:uril", [])
    total_uris = len(uri_list)
    print(f"  [DISCOVERY] 발견된 CIN URI: {total_uris}건")
    if not uri_list:
        return con_list

    # 각 URI에 대해 실제 데이터 GET
    for idx, uri in enumerate(uri_list, start=1):
        print(f"    [FETCH {idx}/{total_uris}] {uri}")
        retries = 0
        while retries < max_retries:
            cin = http_get(base_path + uri, iotPlatform=True)
            if cin is not None:
                con_list.append(cin["m2m:cin"]["con"])
                break
            else:
                retries += 1
                print(f"[Retry {retries}] for URL: {base_path + uri}")

        if retries == max_retries:
            print(f"[FAIL] Data not fetched after {max_retries} attempts for URL: {base_path + uri}")

    return con_list

def fetch_data(base_url):
    """
    base_url 하위의 각 컨테이너(CONTAINERS)로부터 데이터를 가져와 dict 형태로 반환한다.
    ground_truth 컨테이너는 weather 필드만 추출한다.
    """
    data = {key: [] for key in CONTAINERS.keys()}
    for key, container in CONTAINERS.items():
        url = f"{base_url}/{container}"
        print(f"[FETCH] '{key}' 컨테이너 discovery 시작: {url}")
        con_list = all_cin_get_uri(url)
        print(f"[FETCH] '{key}' 컨테이너 수신 완료: {len(con_list)}건")

        # 가져온 con_list를 data dict에 저장
        for con in con_list:
            if key == "ground_truth":
                # 'weather' 필드만 추출
                data[key].append(con['weather'])
            else:
                # date 이외의 필드를 리스트로 묶어서 저장
                data[key].append([con[k] for k in con.keys() if k != "date"])
    return data

# --------------------------------------------------------------------------------
# (6) 데이터 전처리 
# --------------------------------------------------------------------------------
def preprocess_data(data, window_size, scalers=None, encoder=None):
    """
    data(dict)에서 precipitation, temperature, wind, ground_truth를 가져와
    - MinMaxScaler로 스케일링
    - weather(문자열)에 대해 OneHotEncoder 적용
    - window_size만큼 시계열 슬라이싱싱
    - scalers=None, encoder=None이면 새로 fit (train)
    - scalers, encoder가 주어지면 transform만 (test)
    """
    # (a) NumPy 배열 변환
    precipitation = np.array(data["precipitation"])
    temperature   = np.array(data["temperature"])
    wind          = np.array(data["wind"])
    weather       = np.array(data["ground_truth"])  # 문자열 배열

    # (b) MinMax 정규화 - fit/transform 분리
    if scalers is None:
        # Train: 각 feature별 scaler를 새로 생성하고 fit_transform
        scaler_precip = MinMaxScaler()
        scaler_temp   = MinMaxScaler()
        scaler_wind   = MinMaxScaler()
        
        precipitation = scaler_precip.fit_transform(precipitation)
        temperature   = scaler_temp.fit_transform(temperature)
        wind          = scaler_wind.fit_transform(wind)
        
        scalers = {
            'precipitation': scaler_precip,
            'temperature':   scaler_temp,
            'wind':          scaler_wind
        }
    else:
        # Test: 기존 scaler로 transform만
        precipitation = scalers['precipitation'].transform(precipitation)
        temperature   = scalers['temperature'].transform(temperature)
        wind          = scalers['wind'].transform(wind)

    # (c) One-hot 인코딩 - fit/transform 분리
    if encoder is None:
        # Train: 새 encoder 생성 및 fit_transform
        encoder = OneHotEncoder(sparse_output=False)
        weather_encoded = encoder.fit_transform(weather.reshape(-1, 1))
        print("[Train] Categories:", encoder.categories_)
    else:
        # Test: 기존 encoder로 transform만
        weather_encoded = encoder.transform(weather.reshape(-1, 1))
        print("[Test] Using encoder with categories:", encoder.categories_)

    # (d) 슬라이딩 윈도우(X, y) 생성
    X, y = [], []
    for i in range(len(precipitation) - window_size):
        features = np.concatenate([
            precipitation[i:i+window_size],
            temperature[i:i+window_size],
            wind[i:i+window_size]
        ], axis=1)
        X.append(features)
        y.append(weather_encoded[i+window_size])

    return np.array(X), np.array(y), scalers, encoder

# --------------------------------------------------------------------------------
# (7) 모델 정의 (LSTM)
# --------------------------------------------------------------------------------
class WeatherLSTM(nn.Module):
    """
    입력 시퀀스로부터 날씨 분류(One-hot 형태)를 수행하는 간단한 LSTM 모델.
    """
    def __init__(self, input_size, hidden_size, output_size):
        super(WeatherLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc   = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        """
        x.shape: (batch, seq_len, input_size)
        LSTM 출력 중 마지막 타임스텝(hidden state)을 fully connected로 연결.
        """
        out, _ = self.lstm(x)
        out    = self.fc(out[:, -1, :])  # 마지막 타임스텝의 출력만 사용
        return out

# --------------------------------------------------------------------------------
# (8) 학습 함수 
# --------------------------------------------------------------------------------
def train_model(model, train_loader, criterion, optimizer, num_epochs):
    """
    - model: WeatherLSTM 인스턴스
    - train_loader: 학습용 DataLoader
    - criterion: 손실 함수
    - optimizer: 옵티마이저
    - num_epochs: 에폭 수
    """
    print("[INFO] Starting training...")
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss    = criterion(outputs, torch.argmax(y_batch, dim=1))
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

    # 학습 완료 후 모델 저장
    torch.save(model.state_dict(), "weather_lstm_model.pth")
    print("[INFO] Model trained and saved: weather_lstm_model.pth")

# --------------------------------------------------------------------------------
# (9) 테스트 함수 
# --------------------------------------------------------------------------------
def test_model(model, test_loader):
    """
    - model: 가중치가 로드된 WeatherLSTM 인스턴스
    - test_loader: 테스트용 DataLoader
    """
    model.eval()
    correct = 0
    total   = 0

    print("[INFO] Starting evaluation...")
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs   = model(X_batch)
            predicted = torch.argmax(outputs, dim=1)
            correct  += (predicted == torch.argmax(y_batch, dim=1)).sum().item()
            total    += y_batch.size(0)

    accuracy = correct / total
    print(f"[RESULT] Test Accuracy: {accuracy:.4f}")

# --------------------------------------------------------------------------------
# (10) 메인 실행부 
# --------------------------------------------------------------------------------
if __name__ == "__main__":
    # Step 1: 데이터 Fetch (train / test 모두)
    print("[MAIN] Fetching train data from:", MOBIUS_TRAIN_AE_URL)
    train_data = fetch_data(MOBIUS_TRAIN_AE_URL)
    
    print("[MAIN] Fetching test data from:", MOBIUS_TEST_AE_URL)
    test_data = fetch_data(MOBIUS_TEST_AE_URL)

    # Step 2: 전처리 (train에서 fit, test에서 재사용)
    print("[MAIN] Preprocessing train data (fit scalers & encoder)...")
    X_train, y_train, scalers, encoder = preprocess_data(train_data, window_size)
    
    print("[MAIN] Preprocessing test data (transform only)...")
    X_test, y_test, _, _ = preprocess_data(test_data, window_size, scalers=scalers, encoder=encoder)

    # Step 3: DataLoader 생성
    train_dataset = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32)
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    test_dataset = TensorDataset(
        torch.tensor(X_test, dtype=torch.float32),
        torch.tensor(y_test, dtype=torch.float32)
    )
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Step 4: 모델 / 손실함수 / 옵티마이저 생성
    input_size  = X_train.shape[2]
    output_size = y_train.shape[1]
    print(f"[MAIN] Input size: {input_size}, Output size: {output_size}")
    
    model     = WeatherLSTM(input_size, hidden_size, output_size)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Step 5: 학습
    print("[MAIN] Training model...")
    train_model(model, train_loader, criterion, optimizer, num_epochs)

    # Step 6: 평가 (저장된 가중치 로드 후 테스트)
    print("[MAIN] Testing model with saved weights...")
    model_loaded = WeatherLSTM(input_size, hidden_size, output_size)
    model_path = "weather_lstm_model.pth"
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"모델 파일을 찾을 수 없습니다: {model_path}")
    
    model_loaded.load_state_dict(torch.load(model_path, weights_only=True))
    model_loaded.eval()
    test_model(model_loaded, test_loader)
    
    print("[MAIN] Done.")

# 추론 실습 — 학습한 모델을 실제로 써 보기

위 셀에서 모델을 **만들고** 정확도를 쟀습니다. 모델은 결국 **쓰려고(추론)** 만듭니다.
여기서는 위 셀이 남긴 것(모델 `model_loaded`, 전처리기 `scalers`·`encoder`)을 그대로 써서 예측합니다.

| 단계 | 하는 일 | oneM2M |
|---|---|---|
| ① | 2015년(검증 데이터)에서 하루를 골라 예측하고 실제 날씨와 비교 | 위 셀이 조회해 둔 데이터를 사용 |
| ② | 내가 만든 AE에 3일치 센서값을 **직접 생성**하고, 최신 3개를 조회해 예측 | AE·컨테이너·cin **생성(Create)** + 조회 |

> 모델은 **3일치**(강수량·최고/최저기온·바람)를 보고 **다음 날** 날씨를 맞힙니다(`window_size = 3`).
> 그래서 하루를 예측하려면 그 **전 3일**의 값이 필요합니다.

**먼저 위의 큰 셀을 실행해 두어야 합니다.**

In [ ]:
# --------------------------------------------------------------------------------
# 추론 공통 함수 — ①·② 모두 이 함수로 예측한다
# --------------------------------------------------------------------------------
import datetime as dt

if "model_loaded" not in globals():
    raise RuntimeError("먼저 위의 학습 셀을 실행하세요 — 모델과 전처리기가 아직 없습니다.")

CATEGORIES = [str(c) for c in encoder.categories_[0]]   # ['drizzle', 'fog', 'rain', 'snow', 'sun']

def predict_next(days):
    """
    days: 연속된 3일치 [(precipitation, temp_max, temp_min, wind), ...] — 오래된 날부터
    반환: (예측 날씨, {날씨: 확률})
    학습 때와 **같은 전처리기**(scalers)로 변환해야 한다. 새로 fit하면 숫자의 의미가 바뀐다.
    """
    a = np.array(days, dtype=float)
    if a.shape != (window_size, 4):
        raise ValueError(f"{window_size}일치 × 4개 값이 필요합니다. 받은 모양: {a.shape}")
    features = np.concatenate([
        scalers['precipitation'].transform(a[:, [0]]),
        scalers['temperature'].transform(a[:, 1:3]),
        scalers['wind'].transform(a[:, [3]]),
    ], axis=1)
    x = torch.tensor(features[None], dtype=torch.float32)   # (1, 3일, 4특성)
    model_loaded.eval()
    with torch.no_grad():
        prob = torch.softmax(model_loaded(x), dim=1)[0].numpy()
    return CATEGORIES[prob.argmax()], {c: float(p) for c, p in zip(CATEGORIES, prob)}

def show_prediction(label, prob):
    for c, p in sorted(prob.items(), key=lambda kv: -kv[1]):
        bar = "█" * int(p * 30)
        print(f"    {c:8s} {p:5.1%} {bar}")
    print(f"  → 예측: {label}")

print("[INFER] 준비 완료. 모델이 고를 수 있는 날씨:", CATEGORIES)

## ① 2015년에서 하루 골라 예측하기

`PREDICT_DATE`만 바꿔 가며 실행해 보세요(2015-01-04 ~ 2015-12-31).

In [ ]:
# ★ 바꿔 볼 곳: 예측할 날짜
PREDICT_DATE = "2015-06-13"

# 검증 데이터(test_data)는 2015-01-01부터 하루에 하나씩 순서대로 들어 있다.
# 그래서 날짜 → 몇 번째 값인지를 날짜 차이로 계산할 수 있다.
TEST_START = dt.date(2015, 1, 1)
target = dt.date.fromisoformat(PREDICT_DATE)
i = (target - TEST_START).days
if not (window_size <= i < len(test_data['ground_truth'])):
    raise ValueError(f"{PREDICT_DATE}는 예측할 수 없습니다 — 전 3일이 2015년 안에 있어야 합니다(2015-01-04 ~ 2015-12-31).")

days = [(test_data['precipitation'][j][0], *test_data['temperature'][j], test_data['wind'][j][0])
        for j in range(i - window_size, i)]

print(f"[INFER] {PREDICT_DATE}의 날씨를 그 전 3일로 예측")
for k, d in enumerate(days):
    print(f"  {target - dt.timedelta(days=window_size - k)}  강수량 {d[0]:5.1f}  최고 {d[1]:5.1f}  최저 {d[2]:5.1f}  바람 {d[3]:4.1f}")

label, prob = predict_next(days)
show_prediction(label, prob)
actual = test_data['ground_truth'][i]
print(f"  → 실제: {actual}   {'✅ 맞힘' if label == actual else '❌ 틀림'}")

### 생각해 볼 것

여러 날짜를 넣어 보면 예측이 **`rain`과 `sun` 두 가지**만 나온다는 걸 알게 됩니다.
2015년에는 `fog`인 날이 51일이나 있는데도 그렇습니다. 아래 셀로 직접 세어 보세요.

In [ ]:
# 2015년 전체를 예측해서, 실제 날씨별로 몇 번 맞혔는지 센다
from collections import Counter

with torch.no_grad():
    pred_idx = model_loaded(torch.tensor(X_test, dtype=torch.float32)).argmax(dim=1).numpy()
true_idx = y_test.argmax(axis=1)

print("예측한 날씨 분포:", dict(Counter(CATEGORIES[k] for k in pred_idx)))
print(f"{'실제 날씨':10s} {'일수':>4s} {'맞힘':>4s}")
for k, c in enumerate(CATEGORIES):
    n = int((true_idx == k).sum())
    hit = int(((true_idx == k) & (pred_idx == k)).sum())
    print(f"{c:10s} {n:4d} {hit:4d}")

학습 데이터(2012~2014)의 대부분이 `rain`·`sun`이라, 모델은 드문 날씨를 고르지 않는 쪽으로 배웠습니다.
정확도 0.6 남짓은 "두 가지 중 하나를 잘 고른다"는 뜻에 가깝습니다. **데이터가 모델을 결정합니다.**

---

## ② 내 데이터를 oneM2M에 올리고 예측하기

지금까지는 이미 있는 리소스를 **조회**만 했습니다. 이번에는 센서(디바이스)가 하는 일을 직접 해 봅니다.

```
/Mobius
└── TS-live                     ← 내가 만드는 AE
    ├── precipitation_sensor    ← 컨테이너 3개 (학습 데이터와 같은 이름)
    ├── temperature_sensor
    └── wind_sensor             ← 각각에 하루 1개씩 cin
```

`ground_truth`는 만들지 않습니다 — 내일 날씨는 아직 모르니까요. 그걸 모델이 맞히는 겁니다.

**생성 요청은 조회와 무엇이 다른가**
- 방법이 `GET`이 아니라 `POST`이고, **만들 위치(부모)**의 주소로 보냅니다.
- `Content-Type` 헤더에 **무엇을 만드는지**(`ty`)를 적습니다 — AE는 2, 컨테이너는 3, cin은 4.
- 본문에 만들 리소스의 내용을 담습니다.

In [ ]:
# --------------------------------------------------------------------------------
# ②-1 AE 만들기
# --------------------------------------------------------------------------------
LIVE_AE_NAME = "TS-live"
LIVE_AE_URL  = os.path.join(MOBIUS_BASE_URL, LIVE_AE_NAME)
LIVE_CONTAINERS = ["precipitation_sensor", "temperature_sensor", "wind_sensor"]

def m2m_create(parent_url, origin, ty, body, show=True):
    """oneM2M 생성 요청. 이미 있으면(409) None을 돌려준다."""
    headers = {
        'Accept':       'application/json',
        'X-M2M-RI':     'create-1',
        'X-M2M-Origin': origin,
        'X-M2M-RVI':    '3',
        'Content-Type': f'application/json;ty={ty}',
    }
    if show:
        print(f"  POST {parent_url}")
        for k, v in headers.items():
            print(f"    {k}: {v}")
        print(f"    본문: {json.dumps(body, ensure_ascii=False)}")
    r = requests.post(parent_url, json=body, headers=headers, timeout=10)
    if r.status_code == 409:
        return None
    r.raise_for_status()
    return r.json()

# AE를 만들 때 Origin "S"는 "신원을 새로 발급해 달라"는 뜻이다.
# 응답의 aei(AE-ID)가 이 AE의 신원이 되고, 이후 이 AE 밑에 쓰는 요청은 이 신원으로 보낸다.
# 주의: 부모 주소 끝에 "/"가 붙으면(…/Mobius/) 404가 난다 → 떼고 보낸다
res = m2m_create(MOBIUS_BASE_URL.rstrip("/"), "S", 2,
                 {"m2m:ae": {"rn": LIVE_AE_NAME, "api": "Nweather-lstm-lab", "rr": False, "srv": ["3"]}})
if res is not None:
    LIVE_AEI = res["m2m:ae"]["aei"]
    print(f"[CREATE] AE 생성 → 신원(aei): {LIVE_AEI}")
else:
    # 셀을 다시 실행한 경우: 이미 있는 AE를 조회해서 신원을 알아낸다
    LIVE_AEI = http_get(LIVE_AE_URL, iotPlatform=True)["m2m:ae"]["aei"]
    print(f"[CREATE] {LIVE_AE_NAME}는 이미 있습니다 → 기존 신원(aei): {LIVE_AEI}")

In [ ]:
# --------------------------------------------------------------------------------
# ②-2 컨테이너 3개 만들기 — 부모는 방금 만든 AE, Origin은 그 AE의 신원
# --------------------------------------------------------------------------------
for cnt in LIVE_CONTAINERS:
    res = m2m_create(LIVE_AE_URL, LIVE_AEI, 3, {"m2m:cnt": {"rn": cnt}}, show=(cnt == LIVE_CONTAINERS[0]))
    print(f"[CREATE] {LIVE_AE_NAME}/{cnt}: {'생성' if res else '이미 있음'}")

In [ ]:
# --------------------------------------------------------------------------------
# ②-3 3일치 센서값 올리기 — 센서마다 하루에 cin 하나
# --------------------------------------------------------------------------------
# ★ 바꿔 볼 곳: 오래된 날부터 3일. (날짜, 강수량 mm, 최고기온 ℃, 최저기온 ℃, 바람 m/s)
#   예) 영하로 떨어지고 비가 오는 3일   → 무엇이 나올까?
#       맑고 따뜻한 3일(강수량 0)        → ?
LIVE_DAYS = [
    ("2026-11-14", 0.0, 14.0, 4.0, 2.0),
    ("2026-11-15", 2.5, 11.0, 5.0, 3.5),
    ("2026-11-16", 6.0, 10.0, 4.0, 4.5),
]

for date, precip, tmax, tmin, wind in LIVE_DAYS:
    # cin의 내용(con)은 학습 데이터와 같은 모양으로 맞춘다 — 그래야 같은 코드로 읽힌다
    contents = {
        "precipitation_sensor": {"date": date, "precipitation": precip},
        "temperature_sensor":   {"date": date, "temp_max": tmax, "temp_min": tmin},
        "wind_sensor":          {"date": date, "wind": wind},
    }
    for cnt, con in contents.items():
        m2m_create(f"{LIVE_AE_URL}/{cnt}", LIVE_AEI, 4, {"m2m:cin": {"con": con}},
                   show=(date == LIVE_DAYS[0][0] and cnt == LIVE_CONTAINERS[0]))
    print(f"[CREATE] {date} 값 3개 업로드")

In [ ]:
# --------------------------------------------------------------------------------
# ②-4 최신 3일치를 조회해서 예측
# --------------------------------------------------------------------------------
# 학습 때와 같은 discovery(fu=1&ty=4)에 lim=3을 붙여 3개만 받는다.
# mobius4는 discovery 결과를 **최신 것부터** 돌려준다 → 모델은 오래된 날부터 받아야 하므로 뒤집는다.
# (학습 데이터를 날짜 역순으로 적재해 둔 것도 같은 이유다 — 최신순으로 읽으면 날짜순이 된다.)
def latest_cins(cnt, n=window_size):
    uril = http_get(f"{LIVE_AE_URL}/{cnt}?fu=1&ty=4&lim={n}", iotPlatform=True).get("m2m:uril", [])
    root = f"{urlparse(MOBIUS_BASE_URL).scheme}://{urlparse(MOBIUS_BASE_URL).netloc}/"
    cons = [http_get(root + uri, iotPlatform=True)["m2m:cin"]["con"] for uri in uril]
    return list(reversed(cons))   # 최신순 → 날짜순

p, t, w = (latest_cins(c) for c in LIVE_CONTAINERS)
if not (len(p) == len(t) == len(w) == window_size):
    raise RuntimeError(f"센서마다 {window_size}개가 필요합니다. 받은 개수: {len(p)}, {len(t)}, {len(w)} — ②-3을 먼저 실행하세요.")

print(f"[FETCH] {LIVE_AE_NAME}에서 받은 최신 {window_size}일 (날짜순으로 뒤집은 뒤)")
for a, b, c in zip(p, t, w):
    print(f"  {a['date']}  강수량 {a['precipitation']:5.1f}  최고 {b['temp_max']:5.1f}  최저 {b['temp_min']:5.1f}  바람 {c['wind']:4.1f}")

days = [(a['precipitation'], b['temp_max'], b['temp_min'], c['wind']) for a, b, c in zip(p, t, w)]
last = dt.date.fromisoformat(p[-1]['date'])
print(f"[INFER] {last + dt.timedelta(days=1)}의 날씨 예측")
label, prob = predict_next(days)
show_prediction(label, prob)

### 해 볼 것

1. ②-3의 `LIVE_DAYS`를 바꾸고 **②-3 → ②-4**를 다시 실행해 보세요. 최신 3개만 읽으므로 새 값으로 예측합니다.
2. 영하의 날씨를 넣으면 ①에서는 한 번도 안 나온 `snow`가 나올 수 있습니다. 학습 데이터(2012~2014)에는 눈 온 날이 있었기 때문입니다.
3. ②-4에서 `reversed`를 빼고 실행해 보세요. 날짜 순서가 뒤집히면 예측이 어떻게 달라질까요?